In [0]:
# ============================================================
# NOTEBOOK 0b — STAGING INGESTION
# O&G Rig Operations Intelligence Platform
# Truncates staging tables and reloads from API
# No transformation — raw columns only, exact mirror of API
# Run order: first in pipeline — no dependencies
# ============================================================

# CELL 1 — SETUP
import requests
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

API_BASE  = "https://og-rig-api.onrender.com"
PAGE_SIZE = 500

# Create staging schema if not exists
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.staging")

print(f"✓ API base  : {API_BASE}")
print(f"✓ Page size : {PAGE_SIZE}")
print("✓ workspace.staging schema ready")

✓ API base  : https://og-rig-api.onrender.com
✓ Page size : 500
✓ workspace.staging schema ready


In [0]:
import time

def fetch_all_pages(endpoint: str) -> list:
    all_data    = []
    page        = 1
    total_pages = None
    max_retries = 3

    while True:
        url = f"{API_BASE}{endpoint}?page={page}&page_size={PAGE_SIZE}"

        # Retry logic — handles cold start timeouts on Render free tier
        for attempt in range(1, max_retries + 1):
            try:
                resp = requests.get(url, timeout=120)
                resp.raise_for_status()
                break
            except requests.exceptions.ReadTimeout:
                if attempt < max_retries:
                    print(f"  → Timeout on page {page}, attempt {attempt}/{max_retries} — retrying in 15s...")
                    time.sleep(15)
                else:
                    raise Exception(
                        f"API timed out after {max_retries} attempts "
                        f"for {endpoint} page {page} — aborting"
                    )

        payload = resp.json()

        if total_pages is None:
            total_pages = payload["total_pages"]
            total_recs  = payload["total_records"]
            print(f"  → {total_recs:,} records across {total_pages} pages")

        all_data.extend(payload["data"])

        if not payload["has_next"]:
            break

        page += 1
        if page % 10 == 0:
            print(f"  → fetched page {page}/{total_pages}...")

    if not all_data:
        raise Exception(
            f"API returned 0 records for {endpoint} — "
            f"aborting to prevent empty staging tables"
        )

    return all_data

print("✓ Fetch helper ready")

✓ Fetch helper ready


In [0]:
# CELL 3 — TRUNCATE STAGING TABLES
# Only truncates if table already exists
# On first run tables don't exist yet — created on write

staging_tables = [
    "stg_rig_readings",
    "stg_equipment_events",
    "stg_maintenance",
    "stg_crew",
    "stg_incidents",
]

for table in staging_tables:
    exists = spark.sql(f"""
        SELECT COUNT(*) as cnt
        FROM information_schema.tables
        WHERE table_catalog = 'workspace'
          AND table_schema  = 'staging'
          AND table_name    = '{table}'
    """).collect()[0]["cnt"]

    if exists:
        spark.sql(f"TRUNCATE TABLE workspace.staging.{table}")
        print(f"✓ Truncated workspace.staging.{table}")
    else:
        print(f"→ workspace.staging.{table} does not exist yet — will be created on load")

print("\n✓ Staging tables ready for fresh load")

✓ Truncated workspace.staging.stg_rig_readings
✓ Truncated workspace.staging.stg_equipment_events
✓ Truncated workspace.staging.stg_maintenance
✓ Truncated workspace.staging.stg_crew
✓ Truncated workspace.staging.stg_incidents

✓ Staging tables ready for fresh load


In [0]:
# CELL 4 — STG_RIG_READINGS
# Source  : /scada/rig-readings
# Grain   : rig_id + recorded_at
# 17,620 records across 36 pages

print("Loading stg_rig_readings...")

schema = StructType([
    StructField("rig_id",          StringType(),  True),
    StructField("recorded_at",     StringType(),  True),
    StructField("well_name",       StringType(),  True),
    StructField("well_type",       StringType(),  True),
    StructField("status",          StringType(),  True),
    StructField("downtime_reason", StringType(),  True),
    StructField("cost_category",   StringType(),  True),
    StructField("drilling_hours",  DoubleType(),  True),
    StructField("downtime_hours",  DoubleType(),  True),
    StructField("npt_hours",       DoubleType(),  True),
    StructField("rop_ft_hr",       DoubleType(),  True),
    StructField("cost_per_day",    DoubleType(),  True),
    StructField("sla_met",         IntegerType(), True),
])

data = fetch_all_pages("/scada/rig-readings")
df   = spark.createDataFrame(data, schema=schema)
df   = df.withColumn("loaded_at", current_timestamp())

(df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.staging.stg_rig_readings"))

count = spark.sql("SELECT COUNT(*) as n FROM workspace.staging.stg_rig_readings").collect()[0]["n"]
if count == 0:
    raise Exception("stg_rig_readings wrote 0 rows — aborting")
print(f"✓ workspace.staging.stg_rig_readings: {count:,} rows")

Loading stg_rig_readings...
  → 17,620 records across 36 pages
  → fetched page 10/36...
  → fetched page 20/36...
  → fetched page 30/36...
✓ workspace.staging.stg_rig_readings: 17,620 rows


In [0]:
# CELL 5 — STG_EQUIPMENT_EVENTS
# Source  : /scada/equipment-events
# Grain   : event_id
# 88,100 records across 177 pages
# Note    : failure_type is nullable — null when no failure

print("Loading stg_equipment_events...")

schema = StructType([
    StructField("event_id",            StringType(),  True),
    StructField("rig_id",              StringType(),  True),
    StructField("recorded_at",         StringType(),  True),
    StructField("equipment_type",      StringType(),  True),
    StructField("failure_flag",        IntegerType(), True),
    StructField("failure_type",        StringType(),  True),   # nullable
    StructField("downtime_caused_hrs", DoubleType(),  True),
    StructField("sla_met",             IntegerType(), True),
    StructField("resolution_hrs",      DoubleType(),  True),
])

data = fetch_all_pages("/scada/equipment-events")
df   = spark.createDataFrame(data, schema=schema)
df   = df.withColumn("loaded_at", current_timestamp())

(df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.staging.stg_equipment_events"))

count = spark.sql("SELECT COUNT(*) as n FROM workspace.staging.stg_equipment_events").collect()[0]["n"]
if count == 0:
    raise Exception("stg_equipment_events wrote 0 rows — aborting")
print(f"✓ workspace.staging.stg_equipment_events: {count:,} rows")

Loading stg_equipment_events...
  → 88,100 records across 177 pages
  → fetched page 10/177...
  → fetched page 20/177...
  → fetched page 30/177...
  → fetched page 40/177...
  → fetched page 50/177...
  → fetched page 60/177...
  → fetched page 70/177...
  → fetched page 80/177...
  → fetched page 90/177...
  → fetched page 100/177...
  → fetched page 110/177...
  → fetched page 120/177...
  → fetched page 130/177...
  → fetched page 140/177...
  → fetched page 150/177...
  → fetched page 160/177...
  → fetched page 170/177...
✓ workspace.staging.stg_equipment_events: 88,100 rows


In [0]:
# CELL 6 — STG_MAINTENANCE
# Source  : /sap-pm/maintenance
# Grain   : maintenance_id
# 8,672 records across 18 pages

print("Loading stg_maintenance...")

schema = StructType([
    StructField("maintenance_id",   StringType(),  True),
    StructField("rig_id",           StringType(),  True),
    StructField("recorded_at",      StringType(),  True),
    StructField("equipment_type",   StringType(),  True),
    StructField("maintenance_type", StringType(),  True),
    StructField("contractor_name",  StringType(),  True),
    StructField("duration_hrs",     DoubleType(),  True),
    StructField("cost",             DoubleType(),  True),
    StructField("technician_count", IntegerType(), True),
])

data = fetch_all_pages("/sap-pm/maintenance")
df   = spark.createDataFrame(data, schema=schema)
df   = df.withColumn("loaded_at", current_timestamp())

(df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.staging.stg_maintenance"))

count = spark.sql("SELECT COUNT(*) as n FROM workspace.staging.stg_maintenance").collect()[0]["n"]
if count == 0:
    raise Exception("stg_maintenance wrote 0 rows — aborting")
print(f"✓ workspace.staging.stg_maintenance: {count:,} rows")

Loading stg_maintenance...
  → 8,672 records across 18 pages
  → fetched page 10/18...
✓ workspace.staging.stg_maintenance: 8,672 rows


In [0]:
# CELL 7 — STG_CREW
# Source  : /erp/crew
# Grain   : rig_id + crew_id + recorded_at
# 35,240 records across 71 pages
# Note    : two rows per rig per day — Day shift and Night shift

print("Loading stg_crew...")

schema = StructType([
    StructField("rig_id",         StringType(),  True),
    StructField("recorded_at",    StringType(),  True),
    StructField("crew_id",        StringType(),  True),
    StructField("shift",          StringType(),  True),
    StructField("hours_worked",   DoubleType(),  True),
    StructField("overtime_hours", DoubleType(),  True),
    StructField("is_present",     IntegerType(), True),
    StructField("npt_attributed", IntegerType(), True),
])

data = fetch_all_pages("/erp/crew")
df   = spark.createDataFrame(data, schema=schema)
df   = df.withColumn("loaded_at", current_timestamp())

(df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.staging.stg_crew"))

count = spark.sql("SELECT COUNT(*) as n FROM workspace.staging.stg_crew").collect()[0]["n"]
if count == 0:
    raise Exception("stg_crew wrote 0 rows — aborting")
print(f"✓ workspace.staging.stg_crew: {count:,} rows")

Loading stg_crew...
  → 35,240 records across 71 pages
  → fetched page 10/71...
  → fetched page 20/71...
  → fetched page 30/71...
  → fetched page 40/71...
  → fetched page 50/71...
  → fetched page 60/71...
  → fetched page 70/71...
✓ workspace.staging.stg_crew: 35,240 rows


In [0]:
# CELL 8 — STG_INCIDENTS
# Source  : /hse/incidents
# Grain   : incident_id
# 1,465 records across 3 pages
# Note    : equipment_type is nullable — some incidents have no equipment

print("Loading stg_incidents...")

schema = StructType([
    StructField("incident_id",         StringType(),  True),
    StructField("rig_id",              StringType(),  True),
    StructField("recorded_at",         StringType(),  True),
    StructField("crew_id",             StringType(),  True),
    StructField("incident_type",       StringType(),  True),
    StructField("equipment_type",      StringType(),  True),   # nullable
    StructField("incident_status",     StringType(),  True),
    StructField("downtime_caused_hrs", DoubleType(),  True),
    StructField("is_recordable",       IntegerType(), True),
])

data = fetch_all_pages("/hse/incidents")
df   = spark.createDataFrame(data, schema=schema)
df   = df.withColumn("loaded_at", current_timestamp())

(df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.staging.stg_incidents"))

count = spark.sql("SELECT COUNT(*) as n FROM workspace.staging.stg_incidents").collect()[0]["n"]
if count == 0:
    raise Exception("stg_incidents wrote 0 rows — aborting")
print(f"✓ workspace.staging.stg_incidents: {count:,} rows")

Loading stg_incidents...
  → 1,465 records across 3 pages
✓ workspace.staging.stg_incidents: 1,465 rows


In [0]:
# CELL 9 — VERIFICATION
# Row counts, column counts, loaded_at check
# Raises exception if any table is empty

tables = [
    "stg_rig_readings",
    "stg_equipment_events",
    "stg_maintenance",
    "stg_crew",
    "stg_incidents",
]

print("=" * 65)
print("STAGING VERIFICATION")
print("=" * 65)
print(f"{'Table':<30} {'Rows':>10}  {'Cols':>6}  {'loaded_at':>10}")
print("-" * 65)

total  = 0
issues = []

for t in tables:
    df         = spark.table(f"workspace.staging.{t}")
    count      = df.count()
    cols       = len(df.columns)
    has_loaded = "loaded_at" in df.columns
    total     += count

    if count == 0:
        issues.append(f"  ⚠ workspace.staging.{t} is empty")
    if not has_loaded:
        issues.append(f"  ⚠ workspace.staging.{t} missing loaded_at column")

    print(f"workspace.staging.{t:<13} {count:>10,}  {cols:>6}  "
          f"{'✓' if has_loaded else '✗':>10}")

print("-" * 65)
print(f"{'Total rows':<30} {total:>10,}")

if issues:
    print("\nISSUES FOUND:")
    for i in issues:
        print(i)
    raise Exception(
        "Staging verification failed — "
        "do not proceed to Notebook_0a"
    )
else:
    print("\n✓ All staging tables verified")
    print("✓ Notebook 0b complete — safe to proceed to Notebook_0a")

STAGING VERIFICATION
Table                                Rows    Cols   loaded_at
-----------------------------------------------------------------
workspace.staging.stg_rig_readings     17,620      14           ✓
workspace.staging.stg_equipment_events     88,100      10           ✓
workspace.staging.stg_maintenance      8,672      10           ✓
workspace.staging.stg_crew          35,240       9           ✓
workspace.staging.stg_incidents      1,465      10           ✓
-----------------------------------------------------------------
Total rows                        151,097

✓ All staging tables verified
✓ Notebook 0b complete — safe to proceed to Notebook_0a
